# 🚀 MoE LLM Workbench - Autonomous Software Architect
ဤ Notebook သည် MoE LLM Workbench ကို Colab တွင် အလိုအလျောက် Setup လုပ်ပြီး Run ပေးမည် ဖြစ်သည်။

### ⚙️ Configuration
အောက်ပါ cell တွင် သင်လိုအပ်သော setting များကို ပြောင်းလဲနိုင်ပါသည်။

In [ ]:
# @title 🔧 App Settings
ENABLE_HEAVY_TRAINING = False # @param {type:"boolean"}
RESUME_FROM_CHECKPOINT = True # @param {type:"boolean"}
USE_WANDB = False # @param {type:"boolean"}
WANDB_API_KEY = "" # @param {type:"string"}
PORT = 3000 # @param {type:"integer"}

In [ ]:
# @title 🔑 Optional: WandB Login
if USE_WANDB and WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
elif USE_WANDB:
    print("⚠️ WandB key missing. Logging will be local-only or disabled.")

### 📂 Step 0: Persistence (Optional)
သင်၏ Model Checkpoints များကို Google Drive တွင် သိမ်းဆည်းလိုပါက အောက်ပါ cell ကို Run ပါ။

In [ ]:
# @title 💾 Step 0: Persistence (Connect Google Drive)
USE_GOOGLE_DRIVE = True # @param {type:"boolean"}
import os
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Create folders on Drive
    DRIVE_BASE = '/content/drive/MyDrive/MoE_Workbench'
    os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)
    os.makedirs(f'{DRIVE_BASE}/data', exist_ok=True)
    
    # Symlink Checkpoints
    if os.path.exists('checkpoints') and not os.path.islink('checkpoints'):
        !rm -rf checkpoints
    if not os.path.exists('checkpoints'):
        os.symlink(f'{DRIVE_BASE}/checkpoints', 'checkpoints')
        
    # Symlink Data (Persist massive datasets)
    if os.path.exists('data') and not os.path.islink('data'):
        !mv data/* {DRIVE_BASE}/data/ 2>/dev/null || true
        !rm -rf data
    if not os.path.exists('data'):
        os.symlink(f'{DRIVE_BASE}/data', 'data')

    print("✅ Everything (Data & Checkpoints) is now synced with Google Drive!")

In [ ]:
# @title 🛠️ Step 1: System Setup
import os

# Run the setup script to install dependencies
!chmod +x setup_colab.sh
!./setup_colab.sh

In [ ]:
# @title 🧠 Step 2: Model & Data Preparation
import os
from pathlib import Path

# Check for any available high-quality models first
model_priority = [
    Path("checkpoints/best.pt"),
    Path("checkpoints/moe_final.pt"),
    Path("checkpoints/last.pt"),
    Path("checkpoints/moe_model_complete.pt")
]

found_model = None
for m in model_priority:
    if m.exists():
        found_model = m
        break

if ENABLE_HEAVY_TRAINING:
    print("🏋️ Heavy Training Mode Active...")
    
    # Step A: Data Generation (Resumes automatically if partial file exists)
    print("📊 Ensuring training dataset is ready...")
    !python3 training/generate_large_dataset.py --size_per_domain 5000
    
    # Step B: Heavy Training (Auto-resumes from last.pt or best.pt)
    print("🚀 Launching Training Engine...")
    wandb_flag = "--use_wandb" if USE_WANDB else ""
    !python3 training/train_unified.py {wandb_flag} --epochs 10
else:
    if found_model:
        print(f"✅ Found established model: {found_model}. Skipping heavy training.")
    else:
        print("⚡ No existing model. Building a quick initialization model to start UI...")
        # Ensure minimal data for fast mode
        if not os.path.exists("data/train.jsonl"):
            !python3 training/generate_large_dataset.py --size_per_domain 100
        !python3 training/train_unified.py --fast_mode
        
print("\n🏁 Model Preparation Complete.")


In [ ]:
# @title 🌐 Step 3: Launch Workbench
import time
from google.colab import output

print("📡 Starting Backend (Unified Intelligence Platform)... ")
get_ipython().system_raw('python3 backend/app_unified.py &')

print("🛰️ Starting Frontend (Vite Dev Server)... ")
get_ipython().system_raw('export MOCK_AI=false && npm run dev &')

time.sleep(15)
print("\n🚀 MoE Workbench is LIVE!")
output.proxy_port(PORT)